# Analyse 2: Zeitliche Muster bei Parking Violations FY2023–FY2025

Gibt es zeitliche Muster bei Parking Violations nach Monat, Wochentag und Tageszeit?

In [ ]:
import pyspark.sql.functions as f
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pyspark.sql import SparkSession

# Alte (evtl. tote) Session sauber wegräumen
try:
    SparkSession.builder.getOrCreate().stop()
except Exception:
    pass

# Auch den globalen aktiven Context auf None setzen, damit getOrCreate() wirklich neu baut
from pyspark import SparkContext
SparkContext._active_spark_context = None

# Jetzt frisch aufbauen
spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_TimePatterns") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "18") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Verifizieren, dass der Context wirklich lebt
print("SparkContext active:", not spark.sparkContext._jsc.sc().isStopped())
spark

In [ ]:
# Daten laden
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"
df = spark.read.parquet(processed_path)
df.count()

## Analyse A — Monatliche Verteilung

In [ ]:
# Aggregation: Anzahl Violations pro Fiskaljahr und Fiscal Month
monthly = df.filter("is_complete_fy") \
    .groupBy("fy", "fm") \
    .count() \
    .orderBy("fy", "fm") \
    .toPandas()

# Fiscal-Month-Labels: 1=Jul, ..., 6=Dez, 7=Jan, ..., 12=Jun
fm_labels = {1:"Jul", 2:"Aug", 3:"Sep", 4:"Okt", 5:"Nov", 6:"Dez",
             7:"Jan", 8:"Feb", 9:"Mär", 10:"Apr", 11:"Mai", 12:"Jun"}

# Pivot: Fiscal Months als Zeilen, Fiskaljahre als Spalten
pivot_m = monthly.pivot(index="fm", columns="fy", values="count")
pivot_m.index = pivot_m.index.map(fm_labels)

# Plot: Line-Chart, eine Linie pro Fiskaljahr
fig, ax = plt.subplots(figsize=(12, 5))
pivot_m.plot(marker="o", ax=ax)
ax.set_title("Parking Violations pro Monat nach Fiskaljahr", fontsize=13, fontweight="bold")
ax.set_xlabel("Monat (Fiskaljahr beginnt im Juli)")
ax.set_ylabel("Anzahl Violations")
# Y-Achse in Millionen formatieren (z.B. 1.2M statt 1200000)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(True, alpha=0.4)
ax.legend(title="Fiskaljahr")
plt.tight_layout()
plt.show()

In [ ]:
# Aggregation nach Kalenderjahr und Monat
monthly = df.groupBy("issue_year", "issue_month") \
    .count() \
    .orderBy("issue_year", "issue_month") \
    .toPandas()

month_labels = {1:"Jan", 2:"Feb", 3:"Mär", 4:"Apr", 5:"Mai", 6:"Jun",
                7:"Jul", 8:"Aug", 9:"Sep", 10:"Okt", 11:"Nov", 12:"Dez"}

pivot_m = monthly.pivot(index="issue_month", columns="issue_year", values="count")
pivot_m.index = pivot_m.index.map(month_labels)

fig, ax = plt.subplots(figsize=(12, 5))
pivot_m.plot(marker="o", ax=ax)
ax.set_title("Parking Violations pro Monat nach Kalenderjahr\n(2022 nur Jul–Dez, 2025 nur Jan–Jun)", 
             fontsize=13, fontweight="bold")
ax.set_xlabel("Monat")
ax.set_ylabel("Anzahl Violations")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(True, alpha=0.4)
ax.legend(title="Kalenderjahr")
plt.tight_layout()
plt.show()

In [ ]:
# Duplikat-Check: Gibt es summons_numbers die mehrfach vorkommen?
total = df.count()
unique = df.select("summons_number").distinct().count()
print(f"Total Records  : {total:,}")
print(f"Unique Summons : {unique:,}")
print(f"Duplikate      : {total - unique:,}")

## Analyse B — Wochentag (normalisiert)

In [ ]:
weekday_labels = {1:"So", 2:"Mo", 3:"Di", 4:"Mi", 5:"Do", 6:"Fr", 7:"Sa"}

# Nur vollständige Fiskaljahre (vermeidet Verzerrung durch Halbjahre 2022/2025)
df_wd = df.filter("is_complete_fy")

# Rohzählung pro Wochentag
weekday = df_wd.groupBy("issue_weekday").count() \
    .orderBy("issue_weekday").toPandas()

# Anzahl einzigartiger Tage pro Wochentag (für Normalisierung)
days_per_weekday = df_wd.select("issue_date_parsed", "issue_weekday") \
    .distinct() \
    .groupBy("issue_weekday").count() \
    .withColumnRenamed("count", "num_days") \
    .toPandas()

# Durchschnitt berechnen: Violations / Anzahl Tage dieses Wochentags
weekday = weekday.merge(days_per_weekday, on="issue_weekday")
weekday["avg_per_day"] = weekday["count"] / weekday["num_days"]
weekday["label"] = weekday["issue_weekday"].map(weekday_labels)

# Wochentagsreihenfolge: Mo, Di, Mi, Do, Fr, Sa, So (intuitiver als So-Sa)
weekday_order = [2, 3, 4, 5, 6, 7, 1]
weekday["sort_order"] = weekday["issue_weekday"].map({v: i for i, v in enumerate(weekday_order)})
weekday = weekday.sort_values("sort_order")

# Plot: Balkendiagramm normalisiert
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(weekday["label"], weekday["avg_per_day"], color="#1f77b4")
ax.set_title("Durchschnittliche Violations pro Wochentag (normalisiert)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Wochentag")
ax.set_ylabel("⌀ Violations pro Tag")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
ax.grid(True, axis="y", alpha=0.4)

# Werte auf Balken anzeigen — Offset proportional zum Max statt fix
y_max = weekday["avg_per_day"].max()
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + y_max * 0.01, f"{h/1e3:.1f}k",
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

Analyse C — Tageszeit

In [ ]:
hourly = df.filter("is_complete_fy") \
    .filter(f.col("violation_hour").isNotNull()) \
    .groupBy("violation_hour").count() \
    .orderBy("violation_hour").toPandas()

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hourly["violation_hour"], hourly["count"], color="#1f77b4")

ax.set_title("Violations nach Stunde des Tages", fontsize=13, fontweight="bold")
ax.set_xlabel("Stunde (0–23)")
ax.set_ylabel("Anzahl Violations")
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Aggregation: Wochentag × Stunde (nur volle FYs, nur Records mit gültiger Stunde)
heat = df.filter("is_complete_fy") \
    .filter(f.col("violation_hour").isNotNull()) \
    .groupBy("issue_weekday", "violation_hour") \
    .count() \
    .toPandas()

# Pivot: Wochentage als Zeilen, Stunden als Spalten
pivot_h = heat.pivot(index="issue_weekday", columns="violation_hour", values="count")

# Wochentagsreihenfolge Mo-So (intuitiver als So-Sa, das Spark per Default liefert)
weekday_order = [2, 3, 4, 5, 6, 7, 1]
pivot_h = pivot_h.reindex(weekday_order)
pivot_h.index = pivot_h.index.map(weekday_labels)

# Stunden 0..23 in Reihenfolge
pivot_h = pivot_h.sort_index(axis=1)

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(pivot_h.values, cmap="YlOrRd", aspect="auto")

# Achsenbeschriftung
ax.set_xticks(np.arange(len(pivot_h.columns)))
ax.set_xticklabels(pivot_h.columns)
ax.set_yticks(np.arange(len(pivot_h.index)))
ax.set_yticklabels(pivot_h.index)
ax.set_xlabel("Stunde")
ax.set_ylabel("Wochentag")
ax.set_title("Violations: Wochentag × Stunde", fontsize=13, fontweight="bold")

# Colorbar: in Millionen formatieren
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Anzahl Violations")
cbar.formatter = mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M")
cbar.update_ticks()

plt.tight_layout()
plt.show()

In [ ]:
spark.stop()